In [2]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

Working directory: C:\Users\makar\Python\auto-data-scientist


In [3]:
%pip install mlflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
%%writefile src/tracking.py
import re
from pathlib import Path

import mlflow

from src.orchestrator import RunResult

# Matches lines like "Test accuracy: 0.9737" or "ROC-AUC: 0.99" and captures name + number
METRIC_LINE = re.compile(r"^([A-Za-z][A-Za-z0-9 _\-]{2,40}?):\s*([0-9]*\.?[0-9]+)\s*$", re.MULTILINE)


def extract_metrics(stdout: str) -> dict:
    """Pull simple 'Name: number' lines out of a step's printed output.

    This is a heuristic, not a guarantee — it only catches metrics the coder
    happened to print in a recognizable format. Good enough for tracking trends
    across runs without requiring the coder to follow a strict output contract.
    """
    metrics = {}
    for name, value in METRIC_LINE.findall(stdout):
        # MLflow metric keys can't contain spaces or most punctuation
        key = re.sub(r"[^A-Za-z0-9_]", "_", name.strip().lower())
        try:
            metrics[key] = float(value)
        except ValueError:
            pass
    return metrics


def log_run_to_mlflow(result: RunResult, workspace: Path, goal: str,
                      experiment_name: str = "auto-data-scientist"):
    """Log one full pipeline run as an MLflow run: params, metrics, and artifacts."""
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run():
        mlflow.log_param("goal", goal)
        mlflow.log_param("problem_type", result.plan.problem_type)
        mlflow.log_param("target_column", result.plan.target_column)
        mlflow.log_param("num_steps", len(result.plan.steps))
        mlflow.log_param("success", result.success)
        if not result.success:
            mlflow.log_param("failed_at_step", result.failed_at_step)

        # Log each step's own printed metrics, tagged with the step number
        # so you can tell "step 4's accuracy" apart from "step 5's accuracy"
        for log in result.logs:
            mlflow.log_param(f"step{log.step_id}_title", log.title)
            mlflow.log_metric(f"step{log.step_id}_coder_attempts", log.coder_attempts)
            mlflow.log_metric(f"step{log.step_id}_approved", int(log.approved))
            for key, value in extract_metrics(log.stdout).items():
                mlflow.log_metric(f"step{log.step_id}_{key}", value)

        # Log every file the pipeline produced as an artifact (charts, models, csvs)
        mlflow.log_artifacts(str(workspace))

        run_id = mlflow.active_run().info.run_id
    return run_id

Overwriting src/tracking.py


In [5]:
%%writefile src/report.py
from pathlib import Path

from src.orchestrator import RunResult


def generate_report(result: RunResult, goal: str, workspace: Path, output_path: Path) -> Path:
    """Turn a RunResult into one readable Markdown file."""
    lines = [
        f"# Auto Data Scientist Report",
        f"",
        f"**Goal:** {goal}",
        f"**Problem type:** {result.plan.problem_type}",
        f"**Target column:** {result.plan.target_column}",
        f"**Overall result:** {'SUCCESS' if result.success else f'FAILED at step {result.failed_at_step}'}",
        f"",
        f"## Steps",
    ]

    for log in result.logs:
        status = "PASSED" if log.approved else "FAILED"
        lines += [
            f"",
            f"### Step {log.step_id}: {log.title} — {status}",
            f"",
            f"**Task:** {log.task}",
            f"",
            f"**Coder attempts:** {log.coder_attempts}",
            f"**Reviewer verdict:** {log.review_reason}",
        ]
        if log.leakage_flags:
            lines.append(f"**Leakage flags:** {', '.join(log.leakage_flags)}")
        if log.stdout.strip():
            lines += ["", "**Output:**", "```", log.stdout.strip(), "```"]

    # List any charts produced, so the report links directly to them
    charts = sorted(Path(workspace).glob("*.png"))
    if charts:
        lines += ["", "## Charts", ""]
        for chart in charts:
            lines.append(f"![{chart.stem}]({chart.name})")

    output_path = Path(output_path)
    output_path.write_text("\n".join(lines), encoding="utf-8")
    return output_path

Overwriting src/report.py


In [8]:
from src.tracking import log_run_to_mlflow
from src.report import generate_report

goal = "Predict whether a tumor is malignant or benign, using the 'target' column."

run_id = log_run_to_mlflow(result2, ws2, goal)
print("Logged to MLflow, run_id:", run_id)

report_path = generate_report(result2, goal, ws2, ws2 / "report.md")
print("Report saved to:", report_path)

NameError: name 'result2' is not defined

In [9]:
%whos

Variable            Type           Data/Info
--------------------------------------------
Path                type           <class 'pathlib._local.Path'>
RUNS_DIR            WindowsPath    C:\Users\makar\Python\auto-data-scientist\runs
generate_report     function       <function generate_report at 0x0000019BC0233CE0>
goal                str            Predict whether a tumor i<...>sing the 'target' column.
log_run_to_mlflow   function       <function log_run_to_mlfl<...>ow at 0x0000019BBE4B0EA0>
os                  module         <module 'os' from 'C:\\Pr<...>\\anaconda3\\Lib\\os.py'>
sys                 module         <module 'sys' (built-in)>
ws2                 WindowsPath    C:\Users\makar\Python\aut<...>rt4_full_run_v2\workspace


In [10]:
from src.orchestrator import run_pipeline

result2 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws2,
)

print("Overall success:", result2.success)

[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering/Selection ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The preprocessing steps (variance threshold, correlation drop, and StandardScaler) are fit on the entire dataset before any train‑test split, causing data leakage.
[coder] attempt 1/4: OK

[orchestrator] --- Step 4: Model Training & Comparison ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 5: Model Evaluation & Visualization ---
[coder] attempt 1/4: FAILED
[coder] attempt 2/4: FAILED
[coder] attempt 3/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The code selects the best model by evaluating ROC‑AUC on the test set, which leaks test information into model selection.
[coder] attempt 1/4: OK

[orchestrator] 

In [11]:
ws3 = RUNS_DIR / "part4_full_run_v3" / "workspace"
ws3.mkdir(parents=True, exist_ok=True)

result2 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws3,
)

print("Overall success:", result2.success)

[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering/Selection ---
[coder] attempt 1/4: FAILED
[coder] attempt 2/4: OK
[orchestrator] Reviewer rejected (retry 1/2): Feature selection (RFECV) was run on the entire dataset before the train‑test split, leaking information from the hold‑out set into the model.
[coder] attempt 1/4: OK

[orchestrator] --- Step 4: Model Training & Comparison ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 5: Best Model Evaluation & Charting ---
[coder] attempt 1/4: FAILED
[coder] attempt 2/4: OK

[orchestrator] --- Step 6: Model Persistence ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The code correctly fits the model on the full training data and saves it without using any test data or leaki

In [12]:
from src.tracking import log_run_to_mlflow
from src.report import generate_report

run_id = log_run_to_mlflow(result2, ws2, goal)
print("Logged to MLflow, run_id:", run_id)

report_path = generate_report(result2, goal, ws2, ws2 / "report.md")
print("Report saved to:", report_path)

2026/09/25 13:16:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/25 13:16:11 INFO mlflow.store.db.utils: Updating database tables
2026/09/25 13:16:14 INFO mlflow.tracking.fluent: Experiment with name 'auto-data-scientist' does not exist. Creating a new experiment.


Logged to MLflow, run_id: 2c391178976a4ae0a455079a75722edb
Report saved to: C:\Users\makar\Python\auto-data-scientist\runs\part4_full_run_v2\workspace\report.md


In [13]:
from IPython.display import Markdown, display

with open(ws2 / "report.md", encoding="utf-8") as f:
    content = f.read()

display(Markdown(content))

# Auto Data Scientist Report

**Goal:** Predict whether a tumor is malignant or benign, using the 'target' column.
**Problem type:** classification
**Target column:** target
**Overall result:** FAILED at step 6

## Steps

### Step 1: Data Exploration — PASSED

**Task:** Load the dataset, compute summary statistics, visualize distributions of each feature, and check class balance of the target.

**Coder attempts:** 1
**Reviewer verdict:** The code loads the data, prints basic info, computes summary statistics (including target), visualizes each feature's distribution (excluding target), and reports class balance as required, with no evidence of data leakage.

**Output:**
```
Shape: (569, 31)

First 5 rows:
   mean radius  mean texture  ...  worst fractal dimension  target
0        17.99         10.38  ...                  0.11890       0
1        20.57         17.77  ...                  0.08902       0
2        19.69         21.25  ...                  0.08758       0
3        11.42         20.38  ...                  0.17300       0
4        20.29         14.34  ...                  0.07678       0

[5 rows x 31 columns]

Summary statistics:
                         count        mean  ...          75%         max
mean radius              569.0   14.127292  ...    15.780000    28.11000
mean texture             569.0   19.289649  ...    21.800000    39.28000
mean perimeter           569.0   91.969033  ...   104.100000   188.50000
mean area                569.0  654.889104  ...   782.700000  2501.00000
mean smoothness          569.0    0.096360  ...     0.105300     0.16340
mean compactness         569.0    0.104341  ...     0.130400     0.34540
mean concavity           569.0    0.088799  ...     0.130700     0.42680
mean concave points      569.0    0.048919  ...     0.074000     0.20120
mean symmetry            569.0    0.181162  ...     0.195700     0.30400
mean fractal dimension   569.0    0.062798  ...     0.066120     0.09744
radius error             569.0    0.405172  ...     0.478900     2.87300
texture error            569.0    1.216853  ...     1.474000     4.88500
perimeter error          569.0    2.866059  ...     3.357000    21.98000
area error               569.0   40.337079  ...    45.190000   542.20000
smoothness error         569.0    0.007041  ...     0.008146     0.03113
compactness error        569.0    0.025478  ...     0.032450     0.13540
concavity error          569.0    0.031894  ...     0.042050     0.39600
concave points error     569.0    0.011796  ...     0.014710     0.05279
symmetry error           569.0    0.020542  ...     0.023480     0.07895
fractal dimension error  569.0    0.003795  ...     0.004558     0.02984
worst radius             569.0   16.269190  ...    18.790000    36.04000
worst texture            569.0   25.677223  ...    29.720000    49.54000
worst perimeter          569.0  107.261213  ...   125.400000   251.20000
worst area               569.0  880.583128  ...  1084.000000  4254.00000
worst smoothness         569.0    0.132369  ...     0.146000     0.22260
worst compactness        569.0    0.254265  ...     0.339100     1.05800
worst concavity          569.0    0.272188  ...     0.382900     1.25200
worst concave points     569.0    0.114606  ...     0.161400     0.29100
worst symmetry           569.0    0.290076  ...     0.317900     0.66380
worst fractal dimension  569.0    0.083946  ...     0.092080     0.20750
target                   569.0    0.627417  ...     1.000000     1.00000

[31 rows x 8 columns]

Class balance (target):
target
1    357
0    212
Name: count, dtype: int64
```

### Step 2: Missing Values & Encoding — PASSED

**Task:** Verify that there are no missing values; if any appear, impute them appropriately, and confirm that all features are numeric (no categorical encoding needed).

**Coder attempts:** 1
**Reviewer verdict:** The code correctly identified no missing values, performed median imputation where needed, ensured all feature columns are numeric, and saved the processed dataset.

**Output:**
```
Initial shape: (569, 31)
Total missing values before imputation: 0
Total missing values after imputation: 0

Column dtypes after processing:
mean radius                float64
mean texture               float64
mean perimeter             float64
mean area                  float64
mean smoothness            float64
mean compactness           float64
mean concavity             float64
mean concave points        float64
mean symmetry              float64
mean fractal dimension     float64
radius error               float64
texture error              float64
perimeter error            float64
area error                 float64
smoothness error           float64
compactness error          float64
concavity error            float64
concave points error       float64
symmetry error             float64
fractal dimension error    float64
worst radius               float64
worst texture              float64
worst perimeter            float64
worst area                 float64
worst smoothness           float64
worst compactness          float64
worst concavity            float64
worst concave points       float64
worst symmetry             float64
worst fractal dimension    float64
target                       int64
dtype: object

Processed dataset saved to 'data_processed.csv'.
```

### Step 3: Feature Engineering/Selection — PASSED

**Task:** Create polynomial interaction terms for highly correlated features, then apply a variance threshold or recursive feature elimination to keep the most predictive subset.

**Coder attempts:** 1
**Reviewer verdict:** The pipeline correctly avoids data leakage by fitting all preprocessing and feature selection steps on the training set only, and the output meets the task requirements.

**Output:**
```
Original shapes -> train: (455, 30), test: (114, 30)
Found 21 highly correlated pairs (|corr| > 0.9)
After adding interactions -> train: (455, 51), test: (114, 51)
After variance threshold -> train: (455, 51), test: (114, 51)
RFECV selected 49 features out of 51
Test Accuracy: 0.9737
Test ROC AUC: 0.9957
Saved processed dataset with shape (569, 50) to 'data_processed_selected.csv'
```

### Step 4: Model Training & Comparison — PASSED

**Task:** Split data into train/validation sets, train at least two classifiers (e.g., Logistic Regression and Random Forest), and record their cross‑validated accuracy and ROC‑AUC.

**Coder attempts:** 1
**Reviewer verdict:** The code correctly splits the data, uses pipelines to avoid leakage, reports cross‑validated accuracy and ROC‑AUC for both models, and evaluates on a held‑out validation set; the high scores are realistic for this dataset.

**Output:**
```
Dataset shape: (569, 31)
Target distribution:
target
1    357
0    212
Name: count, dtype: int64
Train shape: (455, 30), Validation shape: (114, 30)

Logistic Regression:
  CV Accuracy = 0.9780 ± 0.0098
  CV ROC‑AUC = 0.9959 ± 0.0050
  Validation Accuracy = 0.9825
  Validation ROC‑AUC = 0.9954
  Model saved to logistic_regression.joblib

Random Forest:
  CV Accuracy = 0.9626 ± 0.0179
  CV ROC‑AUC = 0.9894 ± 0.0086
  Validation Accuracy = 0.9561
  Validation ROC‑AUC = 0.9931
  Model saved to random_forest.joblib
```

### Step 5: Best Model Evaluation & Charting — PASSED

**Task:** Select the model with the highest validation ROC‑AUC, compute confusion matrix, precision, recall, F1‑score, plot the ROC curve, and save the plot as an image file.

**Coder attempts:** 2
**Reviewer verdict:** The code correctly selects the model with the highest validation ROC‑AUC, computes the requested metrics, plots and saves the ROC curve, and there is no evident data leakage.

**Output:**
```
Loaded data from data_processed_selected.csv
Train shape: (455, 49), Validation shape: (114, 49)
Loaded model 'logistic_regression' from logistic_regression.joblib
Loaded model 'random_forest' from random_forest.joblib
Model 'logistic_regression' validation ROC‑AUC: 0.7551
Model 'random_forest' validation ROC‑AUC: 0.9446

Best model: random_forest with ROC‑AUC = 0.9446

Confusion Matrix:
[[ 0 42]
 [ 0 72]]
Precision: 0.6316
Recall:    1.0000
F1‑score:  0.7742

ROC curve saved to roc_curve.png
```

### Step 6: Model Persistence — FAILED

**Task:** Fit the chosen model on the full training data and serialize it to disk using joblib.dump for later inference.

**Coder attempts:** 1
**Reviewer verdict:** The code fits the model only on a train split, not on the full training dataset as the task required.

**Output:**
```
Train shape: (455, 30), Test shape: (114, 30)
Test Accuracy: 0.9561
Test ROC AUC: 0.9931
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.93      0.94        42
           1       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Model saved to final_random_forest.joblib
```

## Charts

![area_error_dist](area_error_dist.png)
![compactness_error_dist](compactness_error_dist.png)
![concave_points_error_dist](concave_points_error_dist.png)
![concavity_error_dist](concavity_error_dist.png)
![correlation_heatmap](correlation_heatmap.png)
![feature_importance](feature_importance.png)
![fractal_dimension_error_dist](fractal_dimension_error_dist.png)
![mean_area_dist](mean_area_dist.png)
![mean_compactness_dist](mean_compactness_dist.png)
![mean_concave_points_dist](mean_concave_points_dist.png)
![mean_concavity_dist](mean_concavity_dist.png)
![mean_fractal_dimension_dist](mean_fractal_dimension_dist.png)
![mean_perimeter_dist](mean_perimeter_dist.png)
![mean_radius_dist](mean_radius_dist.png)
![mean_smoothness_dist](mean_smoothness_dist.png)
![mean_symmetry_dist](mean_symmetry_dist.png)
![mean_texture_dist](mean_texture_dist.png)
![perimeter_error_dist](perimeter_error_dist.png)
![radius_error_dist](radius_error_dist.png)
![roc_curve](roc_curve.png)
![smoothness_error_dist](smoothness_error_dist.png)
![symmetry_error_dist](symmetry_error_dist.png)
![texture_error_dist](texture_error_dist.png)
![worst_area_dist](worst_area_dist.png)
![worst_compactness_dist](worst_compactness_dist.png)
![worst_concave_points_dist](worst_concave_points_dist.png)
![worst_concavity_dist](worst_concavity_dist.png)
![worst_fractal_dimension_dist](worst_fractal_dimension_dist.png)
![worst_perimeter_dist](worst_perimeter_dist.png)
![worst_radius_dist](worst_radius_dist.png)
![worst_smoothness_dist](worst_smoothness_dist.png)
![worst_symmetry_dist](worst_symmetry_dist.png)
![worst_texture_dist](worst_texture_dist.png)

In [14]:
import pickle

with open(ws2 / "run_result.pkl", "wb") as f:
    pickle.dump(result2, f)

In [15]:
import pickle
with open(ws2 / "run_result.pkl", "rb") as f:
    result2 = pickle.load(f)

In [17]:
from src.orchestrator import run_pipeline
from src.config import RUNS_DIR
import pickle

ws4 = RUNS_DIR / "part4_full_run_v4" / "workspace"
ws4.mkdir(parents=True, exist_ok=True)

result_final = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws4,
)

print("Overall success:", result_final.success)

# Save immediately, before doing anything else with it
with open(ws4 / "run_result.pkl", "wb") as f:
    pickle.dump(result_final, f)
print("Saved result_final to disk.")

[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering / Selection ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): Feature selection (RFE) was fit on the entire dataset before cross‑validation, causing information from the test folds to leak into the model.
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 2/2): Feature engineering (identifying highly correlated pairs) was performed on the entire dataset before cross‑validation, leaking information from the test folds into the training process.
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 3/2): The pipeline correctly computes interactions, variance filtering, and RFE within each CV training fold, avoiding leakage, and the reported accuracy is rea

In [18]:
import pickle

with open(ws4 / "run_result.pkl", "rb") as f:   # use whatever workspace path this run used
    result_final = pickle.load(f)

step3_log = result_final.logs[2]   # Step 3 is index 2 (0-indexed)
print("Approved:", step3_log.approved)
print("Reason:", step3_log.review_reason)
print("Leakage flags:", step3_log.leakage_flags)

Approved: False
Reason: The pipeline correctly computes interactions, variance filtering, and RFE within each CV training fold, avoiding leakage, and the reported accuracy is realistic.
Leakage flags: ['Called fit on the full X before splitting, instead of X_train']


In [19]:
print(step3_log.code)

# -*- coding: utf-8 -*-
"""
Corrected feature engineering pipeline:
- Compute correlations and create interaction terms inside each CV training fold (no leakage).
- Apply VarianceThreshold to drop constant features.
- Use RFE to keep the most predictive subset.
- Evaluate with cross‑validation and save the final processed dataset + model.
"""

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import joblib

# ---------- Custom transformer ----------
class InteractionCreator(BaseEstimator, TransformerMixin):
    """
    Identify highly correlated feature pairs (|corr| > thresh) on the training data
    and add their product as a new interaction feature.
    """
    def __init__(self,

In [20]:
from src.agents.reviewer import check_leakage_patterns
print(check_leakage_patterns(step3_log.code))

['Called fit on the full X before splitting, instead of X_train']


In [22]:
from src.agents.reviewer import check_leakage_patterns

# The actual Step 3 code that was wrongly flagged
print(check_leakage_patterns(step3_log.code))
# EXPECTED NOW: [] (empty — no longer a false positive)

# Make sure a REAL leak is still caught even with cross_val_score present
still_bad = """
cross_val_score(pipeline, X, y, cv=5)
scaler.fit_transform(X_test)
"""
print(check_leakage_patterns(still_bad))
# EXPECTED: still catches the X_test fit, since that pattern isn't CV-exempted

['Called fit on the full X before splitting, instead of X_train']
['Fit a scaler on X_test (should only ever call .transform on test data)', 'Called fit (or fit_transform) using X_test']


In [23]:
%%writefile src/agents/reviewer.py
import re

from pydantic import BaseModel, Field

from src.llm import chat


class ReviewResult(BaseModel):
    """The reviewer's verdict on one completed step."""
    approved: bool
    reason: str = Field(description="One or two sentences explaining the verdict")
    leakage_flags: list[str] = Field(default_factory=list)


# Patterns that are near-certain signs of data leakage in scikit-learn style code.
# Each is (compiled regex, human-readable explanation).
LEAKAGE_PATTERNS = [
    (re.compile(r"\.fit\w*\(\s*X\s*(?:,|\))"), "Called fit on the full X before splitting, instead of X_train"),
    (re.compile(r"scaler\.fit\w*\(\s*X_test"), "Fit a scaler on X_test (should only ever call .transform on test data)"),
    (re.compile(r"\.fit\w*\(\s*X_test"), "Called fit (or fit_transform) using X_test"),
]


def check_leakage_patterns(code: str) -> list[str]:
    """Cheap, deterministic scan for the most common leakage mistakes.
    Returns a list of human-readable warnings (empty list = nothing found).
    """
    # A final .fit(X, y) after proper cross-validation is standard practice,
    # not leakage -- cross_val_score already evaluated the model honestly.
    used_cross_validation = bool(re.search(r"cross_val_score\(|cross_validate\(", code))

    flags = []
    for pattern, msg in LEAKAGE_PATTERNS:
        if used_cross_validation and "before splitting" in msg:
            continue   # skip only this specific pattern when CV was used
        if pattern.search(code):
            flags.append(msg)
    return flags


SYSTEM_PROMPT = """You are a strict senior data scientist reviewing a junior colleague's completed step.

You will see: the task they were given, the code they wrote, and what the code printed when run.

Check for:
- Data leakage (fitting scalers/encoders/models on data that includes the test set, or using the
  target column as a feature, or any information from the future/test leaking into training).
- Whether the output actually satisfies what the task asked for.
- Suspicious results (e.g. accuracy or ROC-AUC extremely close to 1.0 on a non-trivial task is often
  a sign of leakage, not a good model).

Reply with ONLY a JSON object with exactly these keys:
- "approved": true or false
- "reason": one or two sentences explaining your verdict
- "leakage_flags": a list of short strings describing any leakage concerns (empty list if none)
"""


def _extract_json_obj(reply: str) -> str:
    reply = reply.strip().strip("`")
    if reply.lower().startswith("json"):
        reply = reply[4:]
    start, end = reply.find("{"), reply.rfind("}")
    return reply[start:end + 1] if start != -1 and end != -1 else reply


def review_step(task: str, code: str, stdout: str) -> ReviewResult:
    """Judge one completed step. Combines a fast pattern check with an LLM judgment."""
    import json

    pattern_flags = check_leakage_patterns(code)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"TASK:\n{task}\n\nCODE:\n{code}\n\nOUTPUT:\n{stdout}\n\n"
            f"Automated pattern scan already found these potential issues (may be empty): {pattern_flags}"
        )},
    ]
    reply = chat(messages)

    try:
        data = json.loads(_extract_json_obj(reply))
        result = ReviewResult(**data)
    except Exception:
        # If the judge's reply is unusable, fail safe: don't silently approve.
        result = ReviewResult(approved=False, reason="Reviewer reply could not be parsed.",
                              leakage_flags=pattern_flags)

    # Merge in anything the pattern scan found that the LLM might have missed
    for flag in pattern_flags:
        if flag not in result.leakage_flags:
            result.leakage_flags.append(flag)
    if pattern_flags:
        result.approved = False   # a hard pattern match always overrides the LLM's opinion

    return result

Overwriting src/agents/reviewer.py


In [24]:
from src.agents.reviewer import check_leakage_patterns

print(check_leakage_patterns(step3_log.code))
# EXPECTED NOW: [] (empty)

still_bad = """
cross_val_score(pipeline, X, y, cv=5)
scaler.fit_transform(X_test)
"""
print(check_leakage_patterns(still_bad))
# EXPECTED: still catches the X_test fit

[]
['Fit a scaler on X_test (should only ever call .transform on test data)', 'Called fit (or fit_transform) using X_test']


In [25]:
from src.agents.coder import solve_step
from src.agents.reviewer import review_step
from src.profiler import quick_profile
from src.config import RUNS_DIR

ws4 = RUNS_DIR / "part4_full_run_v4" / "workspace"   # or whichever workspace this run used
profile = quick_profile(ws4 / "data.csv")

step3_task = result_final.plan.steps[2].task   # Step 3, index 2

outcome = solve_step(step3_task, profile, ws4, max_attempts=1, verbose=True)
print("Coder success:", outcome.success)

review = review_step(step3_task, outcome.code, outcome.result.stdout)
print("Approved:", review.approved)
print("Reason:", review.reason)
print("Leakage flags:", review.leakage_flags)

[coder] attempt 1/1: FAILED
Coder success: False
Approved: True
Reason: The pipeline correctly avoids target leakage, scales only on training data, and follows the task steps; no suspicious results are evident.
Leakage flags: []


In [ ]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())